In [ ]:
%%capture
!pip install pgmpy --quiet # It will take some time ...

## Inference with a Bayesian Network

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from pgmpy.utils import get_example_model

### Loading the Alarm Bayesian Network

In [ ]:
# Load the model (hint: get_example_model).
model = get_example_model("alarm")

print(model)

### Visualiza the Graph

In [ ]:
# Visualization using Graphviz layout (hint: nx.nx_pydot.graphviz_layout).
plt.figure(figsize=(16, 12))
G = nx.DiGraph(model.edges())
nx.draw(
    G,
    pos=nx.nx_pydot.graphviz_layout(G, prog="dot"),
    with_labels=True,
    node_color='lightblue',
    edge_color='black',
    node_size=2000,
    font_size=10,
    font_weight='bold'
)
plt.title("Bayesian Network - NetworkX")
plt.show()

### Exact Inference

In [ ]:
from pgmpy.inference import VariableElimination

# Construct the inference engine.
inference_engine = VariableElimination(model)

#### Compute the marginal probability of P(VENTLUNG)

In [ ]:
# Compute the probability of lung ventilation.
p_ventlung = inference_engine.query(["VENTLUNG"])

print(p_ventlung)

#### Compute joint probability of P(VENTLUNG, INTUBATION)

In [ ]:
# Compute the joint probability of lung ventilation and intubation.
p_ventlung_intubation = inference_engine.query(["VENTLUNG", "INTUBATION"])

print(p_ventlung_intubation)

#### Compute conditional probability of P(VENTLUNG | INTUBATION = ESOPHAGEAL)

In [ ]:
# Compute the probability of lung ventilation knowing that intubation is esophageal.
p_ventlung_given_intubation = inference_engine.query(
    variables = ["VENTLUNG"],
    evidence = {"INTUBATION": "ESOPHAGEAL"}
)

print(p_ventlung_given_intubation)

#### Compute P(VENTLUNG, VENTALV | INTUBATION = ESOPHAGEAL, VENTTUBE = LOW)

In [ ]:
p_vv_ie_vl = inference_engine.query(
    variables = ["VENTLUNG", "VENTALV"],
    evidence = {"INTUBATION": "ESOPHAGEAL", "VENTTUBE": "LOW"}
)

print(p_vv_ie_vl)

#### Get the state with maximum probability from the previous queries

In [ ]:
# MAP(VENTLUNG)
print(inference_engine.map_query(["VENTLUNG"]))
# MAP(VENTLUNG, INTUBATION)
print(inference_engine.map_query(["VENTLUNG", "INTUBATION"]))
# MAP(VENTLUNG | INTUBATION = ESOPHAGEAL)
print(inference_engine.map_query(
    variables = ["VENTLUNG"],
    evidence = {"INTUBATION": "ESOPHAGEAL"}
))
# MAP(VENTLUNG, VENTALV | INTUBATION = ESOPHAGEAL, VENTUBE = LOW)
print(inference_engine.map_query(
    variables = ["VENTLUNG", "VENTALV"],
    evidence = {"INTUBATION": "ESOPHAGEAL", "VENTTUBE": "LOW"}
))

### Approximate Inference

In [ ]:
from pgmpy.inference import ApproxInference

# Construct the inference engine.
approx_inference_engine = ApproxInference(model)

#### Compute P(VENTLUNG) with different samples

In [ ]:
samples = [100, 1_000, 10_000, 100_000]
approx_p_ventlung = []

for n in samples:
    _p = approx_inference_engine.query(
        ["VENTLUNG"],
        n_samples=n,
        seed=42
    )
    print(_p)
    approx_p_ventlung.append(_p)

In [ ]:
# Collect the approximate queries.
all_p_ventlung = pd.DataFrame({
    ("APPROX_" + str(n)): p.values
    for (n, p) in zip(samples, approx_p_ventlung)
})
# Add the exact query.
all_p_ventlung["EXACT"] = p_ventlung.values
# Set the index.
all_p_ventlung.index = p_ventlung.state_names["VENTLUNG"]

all_p_ventlung

In [ ]:
all_p_ventlung.plot.bar(ylim=(0, 1))